# 🚀 SpatialMQA Benchmark Evaluation on Kaggle GPU
### Benchmark Mỏ Neo: SpatialMQA (ACL 2025 Long)
**Mô hình đánh giá:** LLaVA-1.5-7B + Fine-tuned LoRA Adapter (`phuckhangne/llava-1.5-7b-spatialmqa-lora`)

---
### ⚙️ Cài đặt môi trường trên Kaggle (BẮT BUỘC):
1. **Accelerator:** Chọn `GPU T4 x 2` hoặc `GPU P100` (ở panel Settings bên phải).
2. **Internet:** Chuyển sang `Internet: ON` (để tải weights từ Hugging Face Hub và ảnh từ CDN).
3. **Persistence:** `Variables and Files`.


## 1. Kiểm tra phần cứng GPU

In [ ]:
!nvidia-smi

## 2. Clone mã nguồn dự án từ GitHub

In [ ]:
import os
REPO_DIR = "/kaggle/working/temporal-spatio-qa"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/khang1108/temporal-spatio-qa.git {REPO_DIR}

%cd {REPO_DIR}
!git status

## 3. Cài đặt các thư viện cần thiết
* Cài đặt LLaVA official package, peft, accelerate, bitsandbytes

In [ ]:
# 1. Gỡ torchao cũ để tránh lỗi xung đột với peft dispatcher:
!pip uninstall -y torchao

# 2. Cài LLaVA với --no-deps để giữ nguyên PyTorch CUDA của Kaggle:
!pip install --no-deps git+https://github.com/haotian-liu/LLaVA.git

# 3. Cài các thư viện phụ thuộc (ghim peft==0.9.0 theo chuẩn dự án):
!pip install -q einops einops-exts timm shortuuid "peft==0.9.0" bitsandbytes sentencepiece huggingface_hub
print("✓ Cài đặt dependencies hoàn tất!")


## 4. Tải 5,063 ảnh COCO của SpatialMQA (~800 MB)
* Tải trực tiếp từ CDN Hugging Face (mất ~30-60 giây trên mạng Kaggle)

In [ ]:
!python scripts/datasets/download_images.py
!ls data/spatial_mqa/images | wc -l


## 5. Sanity Check (Kiểm tra thử 5 câu đầu tiên)
* Bước này nạp mô hình gốc + LoRA adapter từ Hugging Face Hub và chạy thử 5 câu để đảm bảo toàn bộ pipeline hoạt động trơn tru.

In [ ]:
!python scripts/datasets/run_eval.py \
    --model_path liuhaotian/llava-v1.5-7b \
    --lora_path phuckhangne/llava-1.5-7b-spatialmqa-lora \
    --test_jsonl data/spatial_mqa/test.jsonl \
    --image_dir data/spatial_mqa/images \
    --output_jsonl /kaggle/working/test_sanity_5.jsonl \
    --max_samples 5


## 6. Chạy Đánh giá Toàn bộ Benchmark (1,076 câu test)
* **Thời gian chạy trên Kaggle GPU (T4 / P100):** Khoảng 8 - 12 phút.
* **Tham số generation:**  (chuẩn theo code tác giả ACL 2025).

In [ ]:
!python scripts/datasets/run_eval.py \
    --model_path liuhaotian/llava-v1.5-7b \
    --lora_path phuckhangne/llava-1.5-7b-spatialmqa-lora \
    --test_jsonl data/spatial_mqa/test.jsonl \
    --image_dir data/spatial_mqa/images \
    --output_jsonl /kaggle/working/llava_1.5_lora_test_predictions.jsonl \
    --temperature 0.4


## 7. Tổng hợp Kết quả & So sánh với Paper (ACL 2025 Long)

In [ ]:
import json
import pandas as pd

metrics_file = "/kaggle/working/llava_1.5_lora_test_predictions_metrics.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r", encoding="utf-8") as f:
        res = json.load(f)

    ov = res["overall"]
    ax = res["by_axis"]
    ps = res["by_perspective"]

    data = [
        ["Overall Accuracy", f"{ov['accuracy']:.2f}%", "46.85%", f"{ov['accuracy'] - 46.85:+.2f}%"],
        ["Macro-Precision", f"{ov['precision']:.2f}%", "46.10%", f"{ov['precision'] - 46.10:+.2f}%"],
        ["Macro-Recall", f"{ov['recall']:.2f}%", "44.56%", f"{ov['recall'] - 44.56:+.2f}%"],
        ["Macro-F1", f"{ov['f1']:.2f}%", "45.32%", f"{ov['f1'] - 45.32:+.2f}%"],
        ["Ax (Trục ngang: left/right)", f"{ax['A_x']:.2f}%", "55.71%", f"{ax['A_x'] - 55.71:+.2f}%"],
        ["Ay (Trục chiều sâu: front/behind)", f"{ax['A_y']:.2f}%", "29.64%", f"{ax['A_y'] - 29.64:+.2f}%"],
        ["Az (Trục đứng: above/below)", f"{ax['A_z']:.2f}%", "48.13%", f"{ax['A_z'] - 48.13:+.2f}%"],
        ["Q1 (Rule 1: Camera ngoài cảnh)", f"{ps['Q1_OutOfImage']:.2f}%", "53.14%", f"{ps['Q1_OutOfImage'] - 53.14:+.2f}%"],
        ["Q2 (Rule 2: Nhập vai thứ nhất - Ego)", f"{ps['Q2_FirstPerson']:.2f}%", "40.99%", f"{ps['Q2_FirstPerson'] - 40.99:+.2f}%"],
        ["Q3 (Rule 3: Góc nhìn thứ ba trong ảnh)", f"{ps['Q3_ThirdPerson']:.2f}%", "64.71%", f"{ps['Q3_ThirdPerson'] - 64.71:+.2f}%"]
    ]

    df = pd.DataFrame(data, columns=["Chỉ số", "Kết quả Kaggle Run", "ACL 2025 Paper", "Delta vs Paper"])
    display(df)
else:
    print(f"Chưa tìm thấy {metrics_file}. Hãy chạy cell 6 trước!")


## 8. Xem trực quan một vài mẫu dự đoán (Sample Inspection)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

pred_file = "/kaggle/working/llava_1.5_lora_test_predictions.jsonl"
if os.path.exists(pred_file):
    with open(pred_file, "r", encoding="utf-8") as f:
        preds = [json.loads(line) for line in f][:4]

    for p in preds:
        img_p = os.path.join("data/spatial_mqa/images", p["image"])
        if os.path.exists(img_p):
            im = Image.open(img_p)
            plt.figure(figsize=(4, 4))
            plt.imshow(im)
            plt.axis("off")
            plt.show()
        print(f"Question: {p['question']}")
        print(f"Options:  {p['options']}")
        print(f"Ground Truth: {p['answer']}")
        print(f"Prediction:   {p['output']}")
        print("-" * 60)
